[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/revenue-segment-hierarchy-python.ipynb)

# Parse Hierarchical Revenue Segments from SEC XBRL Filings with Python

Use **edgartools** to extract revenue segment hierarchies from SEC filings -- see how Apple breaks revenue into Products vs Services, how Microsoft splits by business segment, and how to build parent-child trees from XBRL data.

**What you'll learn:**
- Extract revenue segments with parent-child hierarchy
- Use `level`, `parent_concept`, and `parent_abstract_concept` columns
- Control detail level with `view` parameter (standard, detailed, summary)
- Build custom segment trees for analysis
- Understand when to use `Statement.to_dataframe()` vs `xbrl.query()`

## Install edgartools

In [ ]:
!pip install -q -U edgartools

## Setup

The SEC requires all automated tools to identify themselves. Replace the email below with your own.

In [ ]:
from edgar import *

set_identity("your.name@example.com")

## The Problem: Flat Facts vs Hierarchical Statements

SEC filings report revenue broken down by product, service, geography, and business segment. This data is structured as a **tree** in the XBRL presentation linkbase.

If you use `xbrl.query()` to extract facts, you get a flat list with no hierarchy. The `Statement` object preserves the tree structure -- and `to_dataframe()` gives you columns to navigate it.

## Example 1: Apple Revenue Hierarchy

Apple breaks revenue into Products (iPhone, Mac, iPad, Wearables) and Services. Let's extract this tree.

In [ ]:
company = Company("AAPL")
filing = company.get_filings(form="10-K").latest()
xbrl = filing.xbrl()

# Get the income statement -- hierarchy is preserved automatically
income = xbrl.statements.income_statement()
income

## The Hierarchy Columns

Every `to_dataframe()` call returns these hierarchy columns:

| Column | What it tells you |
|--------|-------------------|
| `level` | Nesting depth (0=root, 1=section, 2=line item, 3=sub-item) |
| `abstract` | True for section headers (no data), False for actual values |
| `parent_concept` | Calculation parent -- the metric this rolls up to for math |
| `parent_abstract_concept` | Presentation parent -- the section header this sits under |

In [ ]:
df = income.to_dataframe()

# Show hierarchy columns for the first 15 rows
hierarchy_cols = ['label', 'level', 'abstract', 'parent_concept', 'parent_abstract_concept']
df[hierarchy_cols].head(15)

## Visualize the Tree Structure

Use the `level` column to display the statement as an indented tree:

In [ ]:
# Build an indented tree view from the hierarchy
date_cols = [c for c in df.columns if c.startswith('20')]
latest_period = date_cols[0] if date_cols else None

for _, row in df.iterrows():
    indent = '  ' * int(row['level'])
    label = row['label']
    
    if row['abstract']:
        print(f"{indent}{label}:")
    elif latest_period:
        value = row[latest_period]
        if value and value == value:  # not NaN
            print(f"{indent}{label:50s} ${value:>15,.0f}")
        else:
            print(f"{indent}{label}")

## Extract Revenue Segments Using parent_concept

The `parent_concept` column tells you which line items roll up to a parent. Use it to find all children of a concept:

In [ ]:
# Find the revenue concept
revenue_rows = df[df['label'].str.contains('net sales|revenue', case=False, na=False) & ~df['abstract']]
print("Revenue-related rows:")
print(revenue_rows[['label', 'concept', 'level']].to_string(index=False))
print()

# Find all rows whose parent_concept matches the total revenue concept
total_revenue_concept = revenue_rows.iloc[-1]['concept'] if len(revenue_rows) > 0 else None

if total_revenue_concept:
    segments = df[df['parent_concept'] == total_revenue_concept]
    print(f"\nChildren of '{total_revenue_concept}':")
    if latest_period:
        print(segments[['label', latest_period]].to_string(index=False))

## Detailed View: All Dimensional Breakdowns

The `view` parameter controls how much data appears:
- `"standard"` — face presentation (what you see in the SEC Viewer)
- `"detailed"` — includes all dimensional breakdowns (segments, geography, etc.)
- `"summary"` — non-dimensional totals only

In [ ]:
# Compare row counts across views
for view in ['summary', 'standard', 'detailed']:
    view_df = income.to_dataframe(view=view)
    print(f"{view:10s} view: {len(view_df):>3} rows")

In [ ]:
# Detailed view shows segment breakdowns
df_detailed = income.to_dataframe(view="detailed")

# Filter to dimensional rows (segment data)
dimensional = df_detailed[df_detailed['dimension'].notna()]
print(f"Dimensional rows: {len(dimensional)}")

if not dimensional.empty and latest_period:
    print(dimensional[['label', 'dimension', latest_period]].head(10).to_string(index=False))

## Example 2: Microsoft Business Segments

Microsoft reports revenue by product (Office, Azure, LinkedIn, Gaming, etc.). The detailed view includes all segment data:

In [ ]:
msft = Company("MSFT")
msft_filing = msft.get_filings(form="10-K").latest()
msft_xbrl = msft_filing.xbrl()

msft_income = msft_xbrl.statements.income_statement()
msft_df = msft_income.to_dataframe(view="detailed")

date_cols = [c for c in msft_df.columns if c.startswith('20')]
latest = date_cols[0] if date_cols else None

# Show the revenue section with hierarchy
revenue_section = msft_df[msft_df['level'] <= 3].head(20)
for _, row in revenue_section.iterrows():
    indent = '  ' * int(row['level'])
    label = row['label']
    if row['abstract']:
        print(f"{indent}{label}:")
    elif latest:
        value = row[latest]
        if value and value == value:
            print(f"{indent}{label:50s} ${value:>15,.0f}")

## Build a Custom Segment DataFrame

For analysis, you often want a clean DataFrame of just the segments with their parent relationship:

In [ ]:
import pandas as pd

def extract_segment_tree(df, period_col=None):
    """Extract a clean segment tree from a statement DataFrame."""
    # Get period column
    if period_col is None:
        date_cols = [c for c in df.columns if c.startswith('20')]
        period_col = date_cols[0] if date_cols else None
    
    # Build parent lookup from level column
    rows = []
    parent_stack = {}  # level -> label
    
    for _, row in df.iterrows():
        level = int(row['level'])
        label = row['label']
        parent_stack[level] = label
        
        parent = parent_stack.get(level - 1) if level > 0 else None
        
        entry = {
            'label': label,
            'level': level,
            'parent': parent,
            'is_header': bool(row['abstract']),
            'concept': row['concept'],
        }
        if period_col:
            entry['value'] = row[period_col]
        rows.append(entry)
    
    return pd.DataFrame(rows)

# Build the tree for Apple
tree = extract_segment_tree(df)
print(tree[~tree['is_header']][['label', 'parent', 'value']].head(10).to_string(index=False))

## When to Use Statement vs Query

EdgarTools has two ways to access XBRL data. Use the right one for your task:

| Goal | Use this | Why |
|------|----------|-----|
| Revenue segment tree | `statement.to_dataframe()` | Has `level`, `parent_concept` hierarchy columns |
| Specific fact lookup | `xbrl.query()` | Fast, filtered access to individual facts |
| Dimensional breakdowns | `statement.to_dataframe(view="detailed")` | Includes all segment/geography data |
| Cross-period comparison | `statement.to_dataframe()` | Multiple period columns side by side |
| Custom fact filtering | `xbrl.facts.query().by_dimension(...)` | Full control over fact selection |

**Key insight:** `xbrl.query()` returns flat facts -- no hierarchy. `Statement.to_dataframe()` preserves the presentation tree.

## Quick Reference

```python
from edgar import *
set_identity("your.name@example.com")

# Get a statement with hierarchy
income = Company("AAPL").get_filings(form="10-K").latest().xbrl().statements.income_statement()
df = income.to_dataframe()

# Hierarchy columns
df['level']                      # Nesting depth (0, 1, 2, ...)
df['abstract']                   # True = section header
df['parent_concept']             # Calculation parent (math rollup)
df['parent_abstract_concept']    # Presentation parent (display hierarchy)

# Control detail level
df = income.to_dataframe(view="standard")   # Face presentation
df = income.to_dataframe(view="detailed")   # All segments
df = income.to_dataframe(view="summary")    # Totals only

# Find children of a concept
children = df[df['parent_concept'] == 'us-gaap_Revenue']
```

## What's Next

- [Parse XBRL Financial Data](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/xbrl-financial-data-python.ipynb) -- Browse all XBRL statements and disclosures
- [Extract Revenue and Earnings](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/extract-revenue-earnings-python.ipynb) -- Pull key financial metrics
- [Compare Company Financials](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/compare-company-financials-python.ipynb) -- Cross-company analysis
- [Dimension Handling Guide](https://edgartools.readthedocs.io/en/latest/xbrl/concepts/dimension-handling/) -- Deep dive on dimensional data

**Resources:**
- [EdgarTools Documentation](https://edgartools.readthedocs.io/)
- [GitHub Repository](https://github.com/dgunning/edgartools)
- [PyPI Package](https://pypi.org/project/edgartools/)

---

## Support EdgarTools

If you found this tutorial helpful:

- **Star the repo** -- [github.com/dgunning/edgartools](https://github.com/dgunning/edgartools)
- **Visit edgartools.io** -- [edgartools.io](https://www.edgartools.io/)
- **Report issues** -- [Open an issue](https://github.com/dgunning/edgartools/issues)

*edgartools is free, open-source, and community-driven. No API key or paid subscription required.*